# Experiment 2: Differentiation of Salmonella Serovars in Mixed Cultures

## Overview
Differentiate specific Salmonella serovars (Enteritidis/SE and Mix) within mixed cultures using SERS sensor data.

**Protocol Alignment**: This notebook implements Experiment 2 from the Official Testing Protocol for Fiber Optics SERS Sensor (Turkey Rinsate).

## Objective
Differentiate S. Enteritidis (SE) and Mix serovars in mixed solutions.

## Key Concepts
- **Serovar differentiation**: Distinguishing between different Salmonella serovars in mixed samples
- **Mixed culture analysis**: Analyzing signals from samples containing multiple serovars
- **Classification**: Machine learning approaches to identify and differentiate serovars
- **Signal characteristics**: Identifying unique spectral features for each serovar

## Protocol Requirements
1. ⏳ Load and organize SERS data for mixed serovar samples
2. ⏳ Extract spectral features unique to each serovar
3. ⏳ Develop classification models to differentiate serovars
4. ⏳ Evaluate differentiation performance metrics
5. ⏳ Compare pure vs. mixed culture signals

## Configuration
- **Target Serovars**: Enteritidis (SE) and Mix
- **Sample Type**: Turkey rinsate from Cargill
- **Mixed Cultures**: Various combinations of serovars

## Status
🚧 **Not Yet Implemented** - This notebook is a placeholder for future implementation.


In [ ]:
# Imports and Setup
# --------------------------------------------------------------------------------------------------
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
import warnings

# Project-specific imports
from sensd_sers_analysis.data import load_dataset_by_serotypes, get_signals_matrix, get_raman_shift

warnings.filterwarnings("ignore")

# Plotting style
plt.style.use("default")
sns.set_palette("husl")
plt.rcParams["figure.figsize"] = (12, 8)
plt.rcParams["font.size"] = 12

print("✅ All imports completed successfully")

## 1. Data Loading and Inspection

Load SERS data for ST, SE, and Mix serotypes. We'll filter to similar concentration ranges to ensure fair comparison.

In [ ]:
# Load SERS Data for Serovar Differentiation
# --------------------------------------------------------------------------------------------------

# Define dataset location
data_folder = "../example_data/SERS Data 8 (April 2025) - test/"
signals_folders = ["March testing-Dilutions"]  # Adjust based on available data

# Actual serotypes in dataset: ST, SE, Mix (not Hadar/Muenchen as per protocol)
serotypes = ["ST", "SE", "Mix"]

try:
    # Load data for all three serotypes
    data_df = load_dataset_by_serotypes(data_folder, signals_folders, serotypes)
    print(f"✅ Successfully loaded {len(data_df)} data entries")

    if not data_df.empty:
        print("\n📊 Dataset Structure Analysis:")
        print("=" * 60)
        print(f"Number of data entries: {len(data_df)}")
        print(f"Raman shift points: {len(data_df['raman_shift'].iloc[0])}")

        # Serotype distribution
        serotype_counts = data_df["serotype"].value_counts()
        print("\nSerotype distribution:")
        for sero, count in serotype_counts.items():
            print(f"  {sero}: {count} entries")

        # Concentration distribution
        print("\nConcentration distribution (all serotypes):")
        conc_counts = data_df["concentration"].value_counts().sort_index()
        for conc, count in conc_counts.head(10).items():
            print(f"  {conc:.1f} CFU/mL: {count} entries")
        if len(conc_counts) > 10:
            print(f"  ... and {len(conc_counts) - 10} more concentrations")

        # Concentration by serotype
        print("\nConcentration ranges by serotype:")
        for sero in serotypes:
            sero_data = data_df[data_df["serotype"] == sero]
            if not sero_data.empty:
                concs = sero_data["concentration"].unique()
                print(
                    f"  {sero}: {concs.min():.1f} - {concs.max():.1f} CFU/mL (median: {np.median(concs):.1f})"
                )

except Exception as e:
    print(f"❌ Error loading data: {e}")
    import traceback

    traceback.print_exc()
    data_df = pd.DataFrame()

## 2. Concentration Binning for Fair Comparison

Since concentrations are not perfect integers, we'll bin them into categories to ensure fair comparison across serotypes.

In [ ]:
# Concentration Binning Function
# --------------------------------------------------------------------------------------------------


def bin_concentrations(concentrations: np.ndarray, method: str = "log10_order") -> np.ndarray:
    """
    Bin concentrations into categories for fair comparison.

    Args:
        concentrations: Array of concentration values
        method: Binning method ('log10_order' or 'quantile')

    Returns:
        Array of bin labels
    """
    if method == "log10_order":
        # Group by order of magnitude (e.g., 10^0, 10^1, 10^2, etc.)
        log_concs = np.log10(concentrations + 1e-10)  # Add small value to avoid log(0)
        bins = np.floor(log_concs).astype(int)
        return bins
    elif method == "quantile":
        # Use quantiles to create equal-sized bins
        q25, q50, q75 = np.percentile(concentrations, [25, 50, 75])
        bins = np.zeros_like(concentrations, dtype=int)
        bins[concentrations <= q25] = 0
        bins[(concentrations > q25) & (concentrations <= q50)] = 1
        bins[(concentrations > q50) & (concentrations <= q75)] = 2
        bins[concentrations > q75] = 3
        return bins
    else:
        raise ValueError(f"Unknown binning method: {method}")


# Apply binning
if not data_df.empty:
    data_df["concentration_bin"] = bin_concentrations(
        data_df["concentration"].values, method="log10_order"
    )

    # Find the most common concentration bin for fair comparison
    most_common_bin = data_df["concentration_bin"].mode()[0]
    print("\n📊 Concentration Binning Analysis:")
    print(f"Most common concentration bin: {most_common_bin} (10^{most_common_bin} order)")

    # Filter to most common bin for fair comparison
    filtered_df = data_df[data_df["concentration_bin"] == most_common_bin].copy()
    print(f"\nFiltered to bin {most_common_bin}: {len(filtered_df)} entries")

    print("\nSerotype distribution in filtered data:")
    for sero in serotypes:
        count = len(filtered_df[filtered_df["serotype"] == sero])
        print(f"  {sero}: {count} entries")

    # Update data_df to filtered version
    data_df = filtered_df

## 3. Average Spectra Comparison

Plot average spectra for ST, SE, and Mix to visualize differences.

In [ ]:
# Average Spectra Comparison
# --------------------------------------------------------------------------------------------------

if not data_df.empty:
    raman_shift = get_raman_shift(data_df)

    # Calculate average spectra for each serotype
    fig, axes = plt.subplots(2, 1, figsize=(14, 10))

    colors = {"ST": "#1f77b4", "SE": "#ff7f0e", "Mix": "#2ca02c"}

    # Plot 1: Average spectra with standard deviation
    ax1 = axes[0]
    for sero in serotypes:
        sero_data = data_df[data_df["serotype"] == sero]
        if len(sero_data) > 0:
            signals_matrix = get_signals_matrix(sero_data)
            mean_signal = np.mean(signals_matrix, axis=0)
            std_signal = np.std(signals_matrix, axis=0)

            ax1.plot(
                raman_shift,
                mean_signal,
                label=f"{sero} (n={len(sero_data)})",
                color=colors.get(sero, "gray"),
                linewidth=2,
            )
            ax1.fill_between(
                raman_shift,
                mean_signal - std_signal,
                mean_signal + std_signal,
                alpha=0.2,
                color=colors.get(sero, "gray"),
            )

    ax1.set_xlabel("Raman Shift (cm⁻¹)", fontsize=12)
    ax1.set_ylabel("Signal Intensity", fontsize=12)
    ax1.set_title("Average Spectra: ST vs SE vs Mix", fontsize=14, fontweight="bold")
    ax1.legend(fontsize=11)
    ax1.grid(True, alpha=0.3)

    # Plot 2: Difference spectra (ST - SE, ST - Mix, SE - Mix)
    ax2 = axes[1]
    sero_means = {}
    for sero in serotypes:
        sero_data = data_df[data_df["serotype"] == sero]
        if len(sero_data) > 0:
            signals_matrix = get_signals_matrix(sero_data)
            sero_means[sero] = np.mean(signals_matrix, axis=0)

    if "ST" in sero_means and "SE" in sero_means:
        ax2.plot(
            raman_shift,
            sero_means["ST"] - sero_means["SE"],
            label="ST - SE",
            color="#1f77b4",
            linewidth=2,
        )
    if "ST" in sero_means and "Mix" in sero_means:
        ax2.plot(
            raman_shift,
            sero_means["ST"] - sero_means["Mix"],
            label="ST - Mix",
            color="#ff7f0e",
            linewidth=2,
        )
    if "SE" in sero_means and "Mix" in sero_means:
        ax2.plot(
            raman_shift,
            sero_means["SE"] - sero_means["Mix"],
            label="SE - Mix",
            color="#2ca02c",
            linewidth=2,
        )

    ax2.axhline(y=0, color="black", linestyle="--", linewidth=1, alpha=0.5)
    ax2.set_xlabel("Raman Shift (cm⁻¹)", fontsize=12)
    ax2.set_ylabel("Difference in Signal Intensity", fontsize=12)
    ax2.set_title("Difference Spectra", fontsize=14, fontweight="bold")
    ax2.legend(fontsize=11)
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

    print("✅ Average spectra comparison completed")

## 4. PCA Visualization

Use Principal Component Analysis to visualize if ST, SE, and Mix form distinct clusters.

In [ ]:
# PCA Visualization
# --------------------------------------------------------------------------------------------------

if not data_df.empty and len(data_df) > 3:
    # Prepare data
    signals_matrix = get_signals_matrix(data_df)
    serotype_labels = data_df["serotype"].values

    # Standardize the data
    scaler = StandardScaler()
    signals_scaled = scaler.fit_transform(signals_matrix)

    # Perform PCA
    pca = PCA(n_components=2)
    pca_result = pca.fit_transform(signals_scaled)

    # Create visualization
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    # Plot 1: PCA scatter plot
    ax1 = axes[0]
    for sero in serotypes:
        mask = serotype_labels == sero
        if mask.sum() > 0:
            ax1.scatter(
                pca_result[mask, 0],
                pca_result[mask, 1],
                label=f"{sero} (n={mask.sum()})",
                color=colors.get(sero, "gray"),
                alpha=0.6,
                s=50,
            )

    ax1.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0] * 100:.1f}% variance)", fontsize=12)
    ax1.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1] * 100:.1f}% variance)", fontsize=12)
    ax1.set_title("PCA: ST vs SE vs Mix Differentiation", fontsize=14, fontweight="bold")
    ax1.legend(fontsize=11)
    ax1.grid(True, alpha=0.3)

    # Plot 2: Explained variance
    ax2 = axes[1]
    n_components_to_show = min(10, signals_scaled.shape[1])
    pca_full = PCA(n_components=n_components_to_show)
    pca_full.fit(signals_scaled)

    ax2.bar(
        range(1, n_components_to_show + 1),
        pca_full.explained_variance_ratio_[:n_components_to_show],
    )
    ax2.set_xlabel("Principal Component", fontsize=12)
    ax2.set_ylabel("Explained Variance Ratio", fontsize=12)
    ax2.set_title("PCA Explained Variance", fontsize=14, fontweight="bold")
    ax2.grid(True, alpha=0.3, axis="y")

    plt.tight_layout()
    plt.show()

    print("\n📊 PCA Results:")
    print(f"PC1 explains {pca.explained_variance_ratio_[0] * 100:.2f}% of variance")
    print(f"PC2 explains {pca.explained_variance_ratio_[1] * 100:.2f}% of variance")
    print(
        f"Total explained variance (PC1+PC2): {(pca.explained_variance_ratio_[0] + pca.explained_variance_ratio_[1]) * 100:.2f}%"
    )

    print("✅ PCA visualization completed")

## 5. Classification Model for Serovar Differentiation

Train a classification model to differentiate between ST, SE, and Mix.

In [ ]:
# Classification Model for Serovar Differentiation
# --------------------------------------------------------------------------------------------------

if not data_df.empty and len(data_df) > 10:
    # Prepare data
    signals_matrix = get_signals_matrix(data_df)
    serotype_labels = data_df["serotype"].values

    # Check if we have enough samples for each class
    unique_serotypes = np.unique(serotype_labels)
    print("\n📊 Classification Setup:")
    print(f"Classes: {unique_serotypes}")
    for sero in unique_serotypes:
        count = np.sum(serotype_labels == sero)
        print(f"  {sero}: {count} samples")

    if len(unique_serotypes) >= 2:
        # Split data
        X_train, X_test, y_train, y_test = train_test_split(
            signals_matrix,
            serotype_labels,
            test_size=0.3,
            random_state=42,
            stratify=serotype_labels,
        )

        # Train Random Forest classifier
        clf = RandomForestClassifier(n_estimators=100, random_state=42, max_depth=10)
        clf.fit(X_train, y_train)

        # Predictions
        y_pred = clf.predict(X_test)
        accuracy = accuracy_score(y_test, y_pred)

        # Results
        print("\n📊 Classification Results:")
        print(f"Accuracy: {accuracy * 100:.2f}%")
        print("\nClassification Report:")
        print(classification_report(y_test, y_pred, target_names=unique_serotypes))

        # Confusion matrix
        cm = confusion_matrix(y_test, y_pred, labels=unique_serotypes)

        fig, ax = plt.subplots(figsize=(8, 6))
        sns.heatmap(
            cm,
            annot=True,
            fmt="d",
            cmap="Blues",
            xticklabels=unique_serotypes,
            yticklabels=unique_serotypes,
            ax=ax,
        )
        ax.set_xlabel("Predicted", fontsize=12)
        ax.set_ylabel("Actual", fontsize=12)
        ax.set_title(
            f"Confusion Matrix (Accuracy: {accuracy * 100:.1f}%)", fontsize=14, fontweight="bold"
        )
        plt.tight_layout()
        plt.show()

        print("✅ Classification model completed")
    else:
        print("⚠️ Not enough classes for classification (need at least 2)")
else:
    print("⚠️ Not enough data for classification")